# 🧠 Brain Tumor Segmentation (BraTS 2023) — Notebook Huấn luyện 2 Mô hình

Notebook này hướng dẫn quy trình huấn luyện và đánh giá hai mô hình Deep Learning phân đoạn khối u não 3D:
1. **3D U-Net** (Convolutional Encoder-Decoder Baseline)
2. **Swin UNETR** (Swin Transformer 3D Encoder + UNet Decoder)

Sử dụng thư viện **PyTorch** & **MONAI**, hỗ trợ chạy trên **Google Colab** (GPU T4/P100/A100) hoặc máy cá nhân.


## 1. Cài đặt các thư viện cần thiết


## 2. Thống nhất Môi trường & Khai báo Thư viện


In [2]:
import os
import sys
import time
import math
import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import monai
import monai.transforms as mt
from monai.networks.nets import UNet, SwinUNETR
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU Device: {torch.cuda.get_device_name(0)}")

 CUDA Available: True
 GPU Device: Tesla T4


## 3. Cấu hình Đường dẫn Dữ liệu BraTS 2023


In [3]:
# Đường dẫn dataset
# Nếu chạy trên Colab, mount Google Drive trước khi tìm dataset.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Ưu tiên đường dẫn trong biến môi trường, sau đó thử các vị trí Google Drive phổ biến.
DATA_ROOT_CANDIDATES = [
    os.environ.get("BRATS_DATA_ROOT"),
    os.environ.get("DATA_ROOT"),
    "Datasets/brats2023-gli-dataset/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/SIC_Capstone_2026/data/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/SIC_Capstone_2026/data/BraTS2023/brats2023-gli-dataset/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
]

DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if not candidate:
        continue
    path = Path(candidate)
    case_dirs = sorted([d.name for d in path.iterdir() if d.is_dir()]) if path.exists() else []
    if case_dirs:
        DATA_ROOT = path
        all_cases = case_dirs
        break

if DATA_ROOT is None:
    DATA_ROOT = Path(DATA_ROOT_CANDIDATES[2])
    all_cases = []
    print("Không tìm thấy dataset. Hãy mount Google Drive và set đúng path, ví dụ:")
    print('os.environ["BRATS_DATA_ROOT"] = "/content/drive/MyDrive/.../ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"')

print(f" Data Root Path: {DATA_ROOT.resolve()}")
print(f" Tổng số cases tìm thấy: {len(all_cases)}")
if len(all_cases) > 0:
    print(f"Sample cases: {all_cases[:5]}")

Mounted at /content/drive
 Data Root Path: /content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
 Tổng số cases tìm thấy: 1251
Sample cases: ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000', 'BraTS-GLI-00006-000']


## 4. Định nghĩa BraTS Dataset Loader


In [4]:
class BraTSDataset3D(Dataset):
    """Dataset custom đọc NIfTI MRI 3D (4 channels: T1n, T1c, T2w, T2f + Mask)."""
    MODALITIES = ["t1n", "t1c", "t2w", "t2f"]

    def __init__(self, data_root: Path, case_list: List[str], transforms=None):
        self.data_root = Path(data_root)
        self.case_list = case_list
        self.transforms = transforms

    def __len__(self):
        return len(self.case_list)

    def __getitem__(self, idx):
        case_id = self.case_list[idx]
        case_dir = self.data_root / case_id

        # Load 4 modalities
        mods = []
        for mod in self.MODALITIES:
            file_path = case_dir / f"{case_id}-{mod}.nii.gz"
            nii = nib.load(file_path)
            mods.append(nii.get_fdata())

        image = np.stack(mods, axis=0).astype(np.float32)  # (4, H, W, D)

        # Z-score normalization per channel
        for c in range(4):
            mean = np.mean(image[c])
            std = np.std(image[c])
            if std > 0:
                image[c] = (image[c] - mean) / std

        # Load segmentation mask if available
        seg_file = case_dir / f"{case_id}-seg.nii.gz"
        if seg_file.exists():
            seg_nii = nib.load(seg_file)
            label = seg_nii.get_fdata().astype(np.float32)
            label = np.expand_dims(label, axis=0)  # (1, H, W, D)
        else:
            label = np.zeros((1,) + image.shape[1:], dtype=np.float32)

        data_dict = {"image": torch.from_numpy(image), "label": torch.from_numpy(label), "case_id": case_id}

        if self.transforms:
            data_dict = self.transforms(data_dict)

        return data_dict

# Train / Val Split
np.random.seed(42)
np.random.shuffle(all_cases)
n_val = max(1, int(len(all_cases) * 0.15))
train_cases = all_cases[:-n_val]
val_cases = all_cases[-n_val:]

print(f"✂️ Data Split: Train = {len(train_cases)} cases | Validation = {len(val_cases)} cases")

✂️ Data Split: Train = 1064 cases | Validation = 187 cases


## 5. MONAI Data Augmentation & DataLoader Setup


In [5]:
# Transforms cho Training (Patch size 128x128x128)
train_transforms = mt.Compose([
    mt.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(128, 128, 128),
        pos=1.0, neg=1.0,
        num_samples=2,
        image_key="image"
    ),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    mt.EnsureTyped(keys=["image", "label"]),
])

val_transforms = mt.Compose([
    mt.EnsureTyped(keys=["image", "label"]),
])

# Tạo Datasets
train_dataset = BraTSDataset3D(DATA_ROOT, train_cases, transforms=train_transforms)
val_dataset = BraTSDataset3D(DATA_ROOT, val_cases, transforms=val_transforms)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print("✅ DataLoaders initialized successfully!")

✅ DataLoaders initialized successfully!


## 6. Huấn luyện Mô hình 1: 3D U-Net Baseline


In [ ]:
# 1. Khởi tạo kiến trúc 3D U-Net
unet_model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,  # Background + 3 Tumor classes
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    dropout=0.2,
    norm="instance"
).to(device)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=True)
optimizer_unet = torch.optim.AdamW(unet_model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

num_epochs = 20  # Có thể tăng lên 50-100 khi train trên GPU
val_interval = 2
best_unet_dice = -1.0
unet_history = {"train_loss": [], "val_dice": []}

print("🚀 Bắt đầu huấn luyện 3D U-Net...")

for epoch in range(1, num_epochs + 1):
    unet_model.train()
    epoch_loss = 0.0
    start_t = time.time()

    for batch in train_loader:
        # Handling list of patches from RandCropByPosNegLabeld
        if isinstance(batch, list):
            batch = batch[0]

        imgs = batch["image"].to(device)
        lbls = batch["label"].to(device)

        optimizer_unet.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = unet_model(imgs)
            loss = loss_function(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.step(optimizer_unet)
        scaler.update()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(1, len(train_loader))
    unet_history["train_loss"].append(avg_loss)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] Train Loss: {avg_loss:.4f} | Time: {time.time()-start_t:.1f}s")

    # Validation
    if epoch % val_interval == 0:
        unet_model.eval()
        dice_scores = []
        with torch.no_grad():
            for val_batch in val_loader:
                v_imgs = val_batch["image"].to(device)
                v_lbls = val_batch["label"].to(device)

                # Sliding window inference
                val_outputs = sliding_window_inference(v_imgs, roi_size=(128, 128, 128), sw_batch_size=4, predictor=unet_model)
                val_preds = (torch.softmax(val_outputs, dim=1) > 0.5).float()

                # Calculate simple Dice
                intersection = torch.sum(val_preds * v_lbls)
                dice = (2.0 * intersection + 1e-5) / (torch.sum(val_preds) + torch.sum(v_lbls) + 1e-5)
                dice_scores.append(dice.item())

        mean_dice = float(np.mean(dice_scores)) if len(dice_scores) > 0 else 0.0
        unet_history["val_dice"].append(mean_dice)
        print(f"   ⭐ Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_unet_dice:
            best_unet_dice = mean_dice
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(unet_model.state_dict(), "checkpoints/unet_brats_best.pth")
            print(f"   💾 Saved Best 3D U-Net Checkpoint! Dice = {best_unet_dice:.4f}")

print("✅ Hoàn thành huấn luyện 3D U-Net!")

/tmp/ipykernel_1018/422190270.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


🚀 Bắt đầu huấn luyện 3D U-Net...


/tmp/ipykernel_1018/422190270.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


## 7. Huấn luyện Mô hình 2: Swin UNETR (3D Transformer)


In [ ]:
# 1. Khởi tạo kiến trúc Swin UNETR
swin_model = SwinUNETR(
    img_size=(128, 128, 128),
    in_channels=4,
    out_channels=4,
    feature_size=48,
    use_checkpoint=True,
    spatial_dims=3
).to(device)

optimizer_swin = torch.optim.AdamW(swin_model.parameters(), lr=2e-4, weight_decay=1e-4)
best_swin_dice = -1.0
swin_history = {"train_loss": [], "val_dice": []}

print("🚀 Bắt đầu huấn luyện Swin UNETR...")

for epoch in range(1, num_epochs + 1):
    swin_model.train()
    epoch_loss = 0.0
    start_t = time.time()

    for batch in train_loader:
        if isinstance(batch, list):
            batch = batch[0]

        imgs = batch["image"].to(device)
        lbls = batch["label"].to(device)

        optimizer_swin.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = swin_model(imgs)
            loss = loss_function(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.step(optimizer_swin)
        scaler.update()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(1, len(train_loader))
    swin_history["train_loss"].append(avg_loss)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] Train Loss: {avg_loss:.4f} | Time: {time.time()-start_t:.1f}s")

    # Validation
    if epoch % val_interval == 0:
        swin_model.eval()
        dice_scores = []
        with torch.no_grad():
            for val_batch in val_loader:
                v_imgs = val_batch["image"].to(device)
                v_lbls = val_batch["label"].to(device)

                val_outputs = sliding_window_inference(v_imgs, roi_size=(128, 128, 128), sw_batch_size=2, predictor=swin_model)
                val_preds = (torch.softmax(val_outputs, dim=1) > 0.5).float()

                intersection = torch.sum(val_preds * v_lbls)
                dice = (2.0 * intersection + 1e-5) / (torch.sum(val_preds) + torch.sum(v_lbls) + 1e-5)
                dice_scores.append(dice.item())

        mean_dice = float(np.mean(dice_scores)) if len(dice_scores) > 0 else 0.0
        swin_history["val_dice"].append(mean_dice)
        print(f"   ⭐ Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_swin_dice:
            best_swin_dice = mean_dice
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(swin_model.state_dict(), "checkpoints/swin_unetr_brats_best.pth")
            print(f"   💾 Saved Best Swin UNETR Checkpoint! Dice = {best_swin_dice:.4f}")

print("✅ Hoàn thành huấn luyện Swin UNETR!")

## 8. Đánh giá & So sánh Hiệu năng hai Mô hình


In [ ]:
# Draw Training Loss & Validation Dice Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(unet_history["train_loss"], label="3D U-Net", color="blue", linewidth=2)
axes[0].plot(swin_history["train_loss"], label="Swin UNETR", color="red", linewidth=2)
axes[0].set_title("Training Loss Comparison", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("DiceCE Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# Plot Dice
axes[1].plot(unet_history["val_dice"], label=f"3D U-Net (Best: {best_unet_dice:.4f})", color="blue", marker="o")
axes[1].plot(swin_history["val_dice"], label=f"Swin UNETR (Best: {best_swin_dice:.4f})", color="red", marker="s")
axes[1].set_title("Validation Dice Score Comparison", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Evaluation Step")
axes[1].set_ylabel("Dice Score")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Trực quan hóa Lát cắt Overlay giữa hai Mô hình


In [ ]:
# Select 1 validation sample to visualize
val_sample = val_dataset[0]
img_tensor = val_sample["image"].unsqueeze(0).to(device)
gt_mask = val_sample["label"].squeeze().numpy()

# Run inference
unet_model.eval()
swin_model.eval()
with torch.no_grad():
    u_out = sliding_window_inference(img_tensor, roi_size=(128, 128, 128), sw_batch_size=2, predictor=unet_model)
    s_out = sliding_window_inference(img_tensor, roi_size=(128, 128, 128), sw_batch_size=2, predictor=swin_model)
    
    u_pred = torch.argmax(torch.softmax(u_out, dim=1), dim=1).squeeze().cpu().numpy()
    s_pred = torch.argmax(torch.softmax(s_out, dim=1), dim=1).squeeze().cpu().numpy()

mri_slice = val_sample["image"][0].numpy()[:, :, 80]  # Axial middle slice
gt_slice = gt_mask[:, :, 80]
u_slice = u_pred[:, :, 80]
s_slice = s_pred[:, :, 80]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(mri_slice, cmap="gray")
axes[0].set_title("MRI T1n Input", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(mri_slice, cmap="gray")
axes[1].imshow(np.ma.masked_where(gt_slice == 0, gt_slice), cmap="jet", alpha=0.6)
axes[1].set_title("Ground Truth Mask", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(mri_slice, cmap="gray")
axes[2].imshow(np.ma.masked_where(u_slice == 0, u_slice), cmap="jet", alpha=0.6)
axes[2].set_title("3D U-Net Prediction", fontweight="bold")
axes[2].axis("off")

axes[3].imshow(mri_slice, cmap="gray")
axes[3].imshow(np.ma.masked_where(s_slice == 0, s_slice), cmap="jet", alpha=0.6)
axes[3].set_title("Swin UNETR Prediction", fontweight="bold")
axes[3].axis("off")

plt.tight_layout()
plt.show()
print("🎉 Hoàn thành Huấn luyện và Trực quan hóa trong Notebook!")